# EDA + Easy Data Augmentation untuk FAKTA**Tujuan:**1. **EDA** (Exploratory Data Analysis) — analisis dataset: distribusi label, kata-kata penting, panjang teks2. **Easy Data Augmentation** — nambah data kelas minoritas (valid) supaya lebih balance3. **Feature Importance** — kata mana yang paling membedakan hoax vs valid**Cara Pakai:**1. Buka https://colab.research.google.com/2. Upload notebook ini3. Upload dataset CSV kamu4. Run semua cell dari atas ke bawah5. Dataset hasil augmentasi tersimpan — bisa dipakai di `colab_lstm_training.ipynb`

---## Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn wordcloud scikit-learnimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom collections import Counterprint('Dependencies ready')

---## Upload Dataset

In [ ]:
!mkdir -p data/trainingfrom google.colab import filesprint('Upload file CSV (combined.csv, dataset.csv, dll)...')uploaded = files.upload()for filename in uploaded.keys():    !mv {filename} data/training/    print(f'  Moved {filename} to data/training/')!ls -la data/training/

---## Load Dataset

In [ ]:
from pathlib import Pathcsv_files = list(Path('data/training').glob('*.csv'))print(f'Found {len(csv_files)} CSV file(s)')dfs = []for f in csv_files:    df = pd.read_csv(f)    if 'text' not in df.columns and 'label' not in df.columns:        df = pd.read_csv(f, header=None, names=['text', 'label'])    if 'text' not in df.columns:        for col in ['tweet', 'content', 'article', 'berita']:            if col in df.columns:                df = df.rename(columns={col: 'text'})                break    if 'label' not in df.columns:        for col in ['hoax', 'class', 'is_hoax', 'label']:            if col in df.columns:                df = df.rename(columns={col: 'label'})                break    df['label'] = df['label'].astype(str).str.lower().str.strip()    label_map = {        'hoax': 'hoax', 'false': 'hoax', '1': 'hoax', 'hoaks': 'hoax',        'valid': 'valid', 'true': 'valid', '0': 'valid',        'tidak hoax': 'valid', 'non-hoax': 'valid',        'uncertain': 'uncertain', 'netral': 'uncertain',    }    df['label'] = df['label'].map(label_map).fillna(df['label'])    dfs.append(df[['text', 'label']])    print(f'  {f.name}: {len(df)} samples ({df["label"].value_counts().to_dict()})')combined = pd.concat(dfs, ignore_index=True)combined = combined.dropna(subset=['text', 'label'])combined = combined.drop_duplicates(subset=['text'])combined = combined[combined['text'].str.len() > 10]print(f'\nTotal: {len(combined)} samples')print(combined['label'].value_counts())

---## EDA - Distribusi Label

In [ ]:
plt.figure(figsize=(8, 5))counts = combined['label'].value_counts()colors = ['#ef4444' if l == 'hoax' else '#22c55e' if l == 'valid' else '#facc15' for l in counts.index]bars = plt.bar(range(len(counts)), counts.values, color=colors, edgecolor='white', linewidth=2)plt.xticks(range(len(counts)), [l.upper() for l in counts.index], fontsize=12, fontweight='bold')plt.ylabel('Jumlah', fontsize=12)plt.title('Distribusi Label dalam Dataset', fontsize=14, fontweight='bold', pad=15)for bar, val in zip(bars, counts.values):    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,             f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')total = len(combined)for label, count in counts.items():    pct = count / total * 100    print(f'  {label:12s}: {count:>6,} ({pct:.1f}%)')ratio = counts.get('hoax', 0) / max(counts.get('valid', 1), 1)print(f'\n  Ratio hoax:valid = {ratio:.1f}:1')if ratio > 2:    print('  IMBALANCE! Perlu augmentasi data valid.')else:    print('  Relatif balanced.')plt.tight_layout()plt.savefig('eda_label_distribution.png', dpi=150)plt.show()

---## EDA - Panjang Teks per Label

In [ ]:
combined['text_length'] = combined['text'].str.len()combined['word_count'] = combined['text'].str.split().str.len()fig, axes = plt.subplots(1, 2, figsize=(14, 5))for label in combined['label'].unique():    subset = combined[combined['label'] == label]    color = '#ef4444' if label == 'hoax' else '#22c55e' if label == 'valid' else '#facc15'    axes[0].hist(subset['text_length'], bins=50, alpha=0.5, label=label, color=color, edgecolor='none')    axes[1].hist(subset['word_count'], bins=50, alpha=0.5, label=label, color=color, edgecolor='none')axes[0].set_title('Distribusi Panjang Karakter', fontsize=13, fontweight='bold')axes[0].set_xlabel('Jumlah Karakter')axes[0].set_ylabel('Frekuensi')axes[0].legend()axes[0].grid(axis='y', alpha=0.3)axes[1].set_title('Distribusi Jumlah Kata', fontsize=13, fontweight='bold')axes[1].set_xlabel('Jumlah Kata')axes[1].set_ylabel('Frekuensi')axes[1].legend()axes[1].grid(axis='y', alpha=0.3)plt.tight_layout()plt.savefig('eda_text_length.png', dpi=150)plt.show()print('Statistik panjang teks:')for label in combined['label'].unique():    subset = combined[combined['label'] == label]    print(f'  {label:12s}: avg={subset["word_count"].mean():.0f} kata, '          f'median={subset["word_count"].median():.0f}, '          f'min={subset["word_count"].min()}, '          f'max={subset["word_count"].max()}')

---## EDA - Top Kata per Label (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizerimport redef simple_clean(text):    text = text.lower()    text = re.sub(r'[^a-z\s]', ' ', text)    return textcombined['clean_simple'] = combined['text'].apply(simple_clean)vectorizer = TfidfVectorizer(max_features=2000, stop_words=None, min_df=5)print('Top 20 Kata per Label (berdasarkan TF-IDF score):\n')for label in combined['label'].unique():    subset = combined[combined['label'] == label]    if len(subset) < 10:        continue    tfidf_matrix = vectorizer.fit_transform(subset['clean_simple'].tolist())    feature_names = vectorizer.get_feature_names_out()    scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()    top_idx = scores.argsort()[-20:][::-1]    print(f'=== {label.upper()} ===')    for i in top_idx:        print(f'  {feature_names[i]:20s}  {scores[i]:.1f}')    print()

---## Feature Importance (Hoax vs Valid)

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import train_test_splitdf_binary = combined[combined['label'].isin(['hoax', 'valid'])].copy()X_train_text, _, y_train_labels, _ = train_test_split(    df_binary['clean_simple'], df_binary['label'], test_size=0.3, random_state=42)tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)X_tfidf = tfidf.fit_transform(X_train_text.tolist())clf = LogisticRegression(max_iter=1000, C=1.0)y_binary = np.array([0 if l == 'valid' else 1 for l in y_train_labels])clf.fit(X_tfidf, y_binary)feature_names = tfidf.get_feature_names_out()coefs = clf.coef_[0]top_hoax_idx = coefs.argsort()[-30:][::-1]top_hoax = [(feature_names[i], float(coefs[i])) for i in top_hoax_idx]top_valid_idx = coefs.argsort()[:30]top_valid = [(feature_names[i], float(coefs[i])) for i in top_valid_idx]fig, axes = plt.subplots(1, 2, figsize=(16, 10))hoax_words = [w for w, _ in top_hoax[::-1]]hoax_scores = [s for _, s in top_hoax[::-1]]axes[0].barh(hoax_words, hoax_scores, color='#ef4444', alpha=0.8)axes[0].set_title('Kata Indikator HOAX', fontsize=14, fontweight='bold')axes[0].set_xlabel('Coefficient (positif = hoax)')axes[0].invert_yaxis()axes[0].grid(axis='x', alpha=0.3)valid_words = [w for w, _ in top_valid]valid_scores = [s for _, s in top_valid]axes[1].barh(valid_words, valid_scores, color='#22c55e', alpha=0.8)axes[1].set_title('Kata Indikator VALID', fontsize=14, fontweight='bold')axes[1].set_xlabel('Coefficient (negatif = valid)')axes[1].invert_yaxis()axes[1].grid(axis='x', alpha=0.3)plt.tight_layout()plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')plt.show()fi_df = pd.DataFrame({    'word': feature_names,    'coefficient': [round(float(c), 4) for c in coefs],    'indicator': ['hoax' if c > 0 else 'valid' for c in coefs],}).sort_values('coefficient', ascending=False)fi_df.to_csv('feature_importance.csv', index=False)print(f'Total kata dianalisis: {len(feature_names)}')print(f'Saved to feature_importance.csv')

---## Step 8: Easy Data Augmentation (EDA)**4 Teknik Augmentasi:**1. **Synonym Replacement** — ganti kata dengan sinonim2. **Random Insertion** — tambah kata acak3. **Random Swap** — tukar posisi kata4. **Random Deletion** — hapus beberapa kata

In [ ]:
import randomSYNONYMS = {    "menyebabkan": ["memicu", "mengakibatkan", "menimbulkan"],    "menurut": ["berdasarkan", "mengacu pada", "merujuk"],    "banyak": ["sejumlah", "beragam", "jumlah"],    "besar": ["signifikan", "luas"],    "kecil": ["sedikit", "minimal"],    "penting": ["krusial", "vital", "mendesak"],    "cepat": ["segera", "segera mungkin"],    "lama": ["berkepanjangan", "panjang"],    "baru": ["terkini", "terbaru"],    "baik": ["bagus", "positif", "menguntungkan"],    "buruk": ["negatif", "merugikan", "parah"],    "tinggi": ["naik", "meningkat"],    "rendah": ["turun", "menurun"],    "orang": ["masyarakat", "warga", "penduduk"],    "pemerintah": ["pejabat", "aparatur", "negara"],    "kesehatan": ["medis", "fisik"],    "penyakit": ["gangguan", "keluhan", "masalah kesehatan"],    "sudah": ["telah"],    "tidak": ["belum", "tidak ada"],    "ada": ["terdapat", "ditemukan"],    "kata": ["ucapan", "pernyataan"],    "berita": ["informasi", "kabar", "laporan"],    "meninggal": ["wafat", "meninggal dunia"],    "mati": ["meninggal", "wafat"],    "bahaya": ["ancaman", "risiko"],    "aman": ["terjamin", "selamat"],}def synonym_replacement(text, n=2):    words = text.split()    candidates = [w for w in words if w in SYNONYMS]    if not candidates:        return text    replace_words = random.sample(candidates, min(n, len(candidates)))    for w in replace_words:        synonym = random.choice(SYNONYMS[w])        idx = words.index(w)        words[idx] = synonym    return " ".join(words)def random_insertion(text, n=2):    words = text.split()    candidates = [w for w in words if w in SYNONYMS]    if not candidates:        return text    for _ in range(min(n, len(candidates))):        word = random.choice(candidates)        synonym = random.choice(SYNONYMS[word])        pos = random.randint(0, len(words))        words.insert(pos, synonym)    return " ".join(words)def random_swap(text, n=3):    words = text.split()    if len(words) < 4:        return text    for _ in range(min(n, len(words) // 3)):        i, j = random.sample(range(len(words)), 2)        words[i], words[j] = words[j], words[i]    return " ".join(words)def random_deletion(text, p=0.2):    words = text.split()    if len(words) < 5:        return text    kept = [w for w in words if random.random() > p]    if not kept:        kept = [random.choice(words)]    return " ".join(kept)def augment(text, strategy="all", n_aug=3):    augmented = []    if strategy in ["sr", "all"]:        for _ in range(n_aug):            augmented.append(synonym_replacement(text, n=random.randint(1, 3)))    if strategy in ["ri", "all"]:        for _ in range(n_aug):            augmented.append(random_insertion(text, n=random.randint(1, 2)))    if strategy in ["rs", "all"]:        for _ in range(n_aug):            augmented.append(random_swap(text, n=random.randint(1, 3)))    if strategy in ["rd", "all"]:        for _ in range(n_aug):            augmented.append(random_deletion(text, p=random.uniform(0.1, 0.3)))    return augmented# Testtest = "BMKG mencatat gempa magnitudo 5.2 di Maluku. Warga dihimbau tetap tenang."print("Original:", test)print()for i, aug_text in enumerate(augment(test), 1):    print(f"Augmented {i}:", aug_text)

---## Step 9: Augmentasi Dataset - Tambah Data ValidKita augmentasi kelas **valid** sampai jumlahnya setara dengan **hoax**.

In [ ]:
df_binary = combined[combined["label"].isin(["hoax", "valid"])].copy()hoax_df = df_binary[df_binary["label"] == "hoax"]valid_df = df_binary[df_binary["label"] == "valid"]n_hoax = len(hoax_df)n_valid = len(valid_df)print(f"Sebelum augmentasi:")print(f"  Hoax:  {n_hoax:,}")print(f"  Valid: {n_valid:,}")print(f"  Ratio: {n_hoax/n_valid:.1f}:1")print()n_to_generate = n_hoax - n_validn_per_sample = max(1, n_to_generate // n_valid)n_per_sample = min(n_per_sample, 4)print(f"Perlu {n_to_generate:,} data valid tambahan")print(f"Augmentasi {n_per_sample}x per sample valid")print()random.seed(42)augmented_texts = []augmented_labels = []total = len(valid_df)for idx, row in valid_df.iterrows():    augs = augment(row["text"], strategy="all", n_aug=n_per_sample)    augmented_texts.extend(augs)    augmented_labels.extend(["valid"] * len(augs))    if (idx + 1) % 500 == 0:        print(f"  Progress: {idx+1}/{total} samples processed")augmented_df = pd.DataFrame({"text": augmented_texts, "label": augmented_labels})final_df = pd.concat([hoax_df[["text", "label"]], valid_df[["text", "label"]], augmented_df], ignore_index=True)final_df = final_df.drop_duplicates(subset=["text"])final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)print(f"\nSetelah augmentasi:")print(final_df["label"].value_counts())print(f"\nTotal: {len(final_df):,} samples")n_valid_after = len(final_df[final_df["label"] == "valid"])print(f"Ratio hoax:valid = {len(hoax_df)/n_valid_after:.2f}:1")final_df.to_csv("data/training/augmented_balanced.csv", index=False)print(f"\nDataset tersimpan: data/training/augmented_balanced.csv")

---## Step 10: Download Dataset Hasil AugmentasiFile ini bisa langsung dipakai di `colab_lstm_training.ipynb`.

In [ ]:
from google.colab import filesfiles.download("data/training/augmented_balanced.csv")print("\nDownloaded augmented_balanced.csv")print("Upload file ini ke Colab saat run colab_lstm_training.ipynb")